In [ ]:
import torch
from torch import nn
from d2l import torch as d2l


# Forward RNNScratch  ─┐
#                      ├─ Concatenate → [T, B, 2H]
# Backward RNNScratch ─┘

In [40]:
# Time 방향 반전 확인

time_steps = torch.tensor(
    [1, 2, 3, 4]
)

reversed_time_steps = torch.flip(
    time_steps,
    dims=(0,),
)

print("Forward order:", time_steps)
print("Backward order:", reversed_time_steps)

Forward order: tensor([1, 2, 3, 4])
Backward order: tensor([4, 3, 2, 1])


In [ ]:
# Bidirectional RNN Architecture

class BiRNNScratch(d2l.Module):
    
    def __init__(
        self,
        num_inputs: int,
        num_hiddens: int,
        sigma: float = 1e-2,
    ) -> None:
        super().__init__()
        
        self.num_inputs = num_inputs
        self.single_direction_hiddens = num_hiddens
        
        # Forward와 Backward Output을 Concatenate하므로
        # 최종 Output Dimension은 2H.
        self.num_hiddens = 2 * num_hiddens
        self.sigma = sigma


        # forward_rnn:
        # W_xh⁽ᶠ⁾, W_hh⁽ᶠ⁾, b_h⁽ᶠ⁾
        self.forward_rnn = d2l.RNNScratch(
            num_inputs=num_inputs,
            num_hiddens=num_hiddens,
            sigma=sigma,
        )

        # backward_rnn:
        # W_xh⁽ᵇ⁾, W_hh⁽ᵇ⁾, b_h⁽ᵇ⁾
        self.backward_rnn = d2l.RNNScratch(
            num_inputs=num_inputs,
            num_hiddens=num_hiddens,
            sigma=sigma,
        )

In [ ]:
# Bidirectional Forward Computation

def birnn_scratch_forward(
    self: BiRNNScratch,
    inputs: torch.Tensor,
    states: tuple[
        torch.Tensor,  # forward_state
        torch.Tensor,  # backward_state
    ] | None = None,
) -> tuple[
    list[torch.Tensor],
    tuple[
        torch.Tensor,
        torch.Tensor,
    ],
]:
    
    if states is None:
        forward_state = None
        backward_state = None
    else:
        forward_state, backward_state = states


    # X₁ → X₂ → ... → X_T
    # forward_outputs : list[T × [B, H]]
    # forward_state   : [B, H]
    forward_outputs, forward_state = (
        self.forward_rnn(
            inputs,
            forward_state,
        )
    )
    
    reversed_inputs = torch.flip(
        inputs,
        dims=(0,),
    )
    
    # X_T → ... → X₂ → X₁
    # reversed_backward_outputs : list[T × [B, H]]
    # backward_state            : [B, H]
    reversed_backward_outputs, backward_state = (
        self.backward_rnn(
            reversed_inputs,
            backward_state,
        )
    )
    # Backward Output을 원래 Time 순서로 복원
    backward_outputs = list(
        reversed(
            reversed_backward_outputs
        )
    )
    
    
    # 동일한 Time Step의 Forward/Backward output Concatenate
    # outputs: list[T × [B, 2H]]
    outputs = [
        torch.cat(
            (
                forward_output,  # [B, H]
                backward_output, # [B, H]
            ),
            dim=-1,
        )
        for forward_output, backward_output in zip(
            forward_outputs,
            backward_outputs,
            strict=True,
        )
    ]
    
    return outputs, (
        forward_state,
        backward_state,
    )
    
    
    
setattr(
    BiRNNScratch,
    "forward",
    birnn_scratch_forward,
)

In [ ]:
# Tensor Shape와 Parameter 검증

num_steps = 3
batch_size = 2
num_inputs = 4
num_hiddens = 8

# [T, B, D]:
X = torch.randn(
    num_steps,
    batch_size,
    num_inputs,
)

birnn = BiRNNScratch(
    num_inputs=num_inputs,   # D
    num_hiddens=num_hiddens, # H
)

# outputs: list[T × [B, 2H]]
# states :  ([B, H], [B, H])
outputs, states = birnn(X)

# Y: [T, B, 2H]
Y = torch.stack(
    outputs,
    dim=0,
)

forward_state, backward_state = states

num_parameters = sum(
    parameter.numel()
    for parameter in birnn.parameters()
)


print("Input shape:", tuple(X.shape))
print("Output shape:", tuple(Y.shape))
print(
    "Forward state shape:",
    tuple(forward_state.shape),
)
print(
    "Backward state shape:",
    tuple(backward_state.shape),
)
print("Parameter count:", num_parameters)

Input shape: (3, 2, 4)
Output shape: (3, 2, 16)
Forward state shape: (2, 8)
Backward state shape: (2, 8)
Parameter count: 208
